In [ ]:
# basic implementation of GPT2 language model

import torch
from transformers import AutoTokenizer, GPT2LMHeadModel

In [ ]:
# instantiate tokenizer
tokenizer = AutoTokenizer.from_pretrained('openai-community/gpt2')

# instantiate GPT2 model
model = GPT2LMHeadModel.from_pretrained('openai-community/gpt2')


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
# print model architecture
print(model)

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)


In [ ]:
# look at parameters under the hood in the model
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")

Total Parameters: 124,439,808
Trainable Parameters: 124,439,808


In [ ]:
# print parameters names and shapes
for name, param in model.named_parameters():
    print(f"{name:50} | Shape: {str(param.shape):30} | Trainable: {param.requires_grad}")

transformer.wte.weight                             | Shape: torch.Size([50257, 768])       | Trainable: True
transformer.wpe.weight                             | Shape: torch.Size([1024, 768])        | Trainable: True
transformer.h.0.ln_1.weight                        | Shape: torch.Size([768])              | Trainable: True
transformer.h.0.ln_1.bias                          | Shape: torch.Size([768])              | Trainable: True
transformer.h.0.attn.c_attn.weight                 | Shape: torch.Size([768, 2304])        | Trainable: True
transformer.h.0.attn.c_attn.bias                   | Shape: torch.Size([2304])             | Trainable: True
transformer.h.0.attn.c_proj.weight                 | Shape: torch.Size([768, 768])         | Trainable: True
transformer.h.0.attn.c_proj.bias                   | Shape: torch.Size([768])              | Trainable: True
transformer.h.0.ln_2.weight                        | Shape: torch.Size([768])              | Trainable: True
transformer.h.0.ln_

In [ ]:
# direct parameter modification

# isolate the specific parameters to change (e.g., first token embedding layer; see list above)
embedding_weights = model.transformer.wte.weight

# manually adjust parameters
# use in-place operations with underscore suffix (e.g., .mul_(), .add_(), .copy_()) to update underlying PyTorch data
with torch.no_grad(): # disable gradient calculation / computation graph
    # multiply all weights by a specific scalar factor (e.g., 1.5)
    embedding_weights.mul_(1.5)

    # another example, manually set a specific parameter tensor to a custom tensor
    # new_weights = torch.randn_like(embedding_weights)
    # embedding_weights.copy_(new_weights)

In [ ]:
# set model to evaluation mode
model.eval()

# tokenize input
inputs = tokenizer('Hello, my dog is cute', return_tensors='pt')

# generate token outputs from model
outputs = model.generate(**inputs, max_length=20)

# detokenize output into readable text
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)


[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In [ ]:
print(generated_text)

Hello, my dog is cute. I'm not sure if she's a good dog, but she
